# Standalone Solar PV System Sizing

This walkthrough converts a 24-hour load profile into preliminary PV array, battery and inverter sizing metrics. It is an engineering screening tool, not a final design.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from standalone_pv_design.sizing import (\n    design_system, check_pv_string_configuration, simulate_hourly_dispatch,\n)\nfrom standalone_pv_design.economics import levelized_cost_of_energy

cfg=json.loads((ROOT/"config/design.json").read_text())
load=pd.read_csv(ROOT/"data/demo/load_profile_synthetic.csv")["load_w"].to_numpy()
display(pd.DataFrame({"hour":range(24),"load_w":load}))

## 1. Daily load and peak demand

In [ ]:
daily_energy_kwh=load.sum()/1000
peak_load_w=load.max()
print("Daily energy (kWh):",round(daily_energy_kwh,3))
print("Peak hourly load (W):",peak_load_w)

plt.figure(figsize=(9,3))
plt.plot(range(24),load,marker="o")
plt.xlabel("Hour")
plt.ylabel("Load (W)")
plt.title("Synthetic daily load profile")
plt.grid(alpha=.2)
plt.show()

## 2. Preliminary system sizing

In [ ]:
design=design_system(
    load, cfg["peak_sun_hours"], cfg["autonomy_days"],
    cfg["battery_voltage_v"], cfg["system_derate"],
    cfg["depth_of_discharge"], cfg["battery_efficiency"],
    cfg["power_factor"], cfg["inverter_margin"]
)
pd.Series(design,name="design_value").to_frame()

## Engineering caveats

PV array sizing depends on worst-month solar resource, temperature, shading, tilt, orientation and system losses. Battery sizing depends on chemistry, temperature, ageing, allowable depth of discharge and reliability target. Inverter selection requires motor-starting and surge loads, not just hourly energy. Cable and protection studies require certified equipment data, fault current and applicable electrical standards.

Replace all synthetic inputs with site-specific values before design approval.

## 3. Read the sizing outputs and understand the assumptions

The PV array estimate divides the daily AC energy requirement by peak-sun-hours, a PV-side derate factor and inverter efficiency. The battery estimate additionally accounts for allowed depth of discharge and battery discharge efficiency. These are planning equations, not detailed equipment simulations.

We pass assumptions by name below so it is clear which value affects each calculation.

In [ ]:
design = design_system(
    load_profile_w=load,
    peak_sun_hours=cfg["peak_sun_hours"],
    autonomy_days=cfg["autonomy_days"],
    battery_voltage_v=cfg["battery_voltage_v"],
    system_derate=cfg["system_derate"],
    depth_of_discharge=cfg["depth_of_discharge"],
    battery_efficiency=cfg["battery_efficiency"],
    power_factor=cfg["power_factor"],
    inverter_margin=cfg["inverter_margin"],
    inverter_efficiency=cfg.get("inverter_efficiency", 0.92),
    interval_hours=cfg.get("interval_hours", 1.0),
    charge_controller_safety_factor=cfg.get("charge_controller_safety_factor", 1.25),
)
display(pd.Series(design).to_frame("value"))

## 4. Check an example PV string against controller limits

PV module open-circuit voltage rises as cell temperature falls. The following values are fictional example datasheet parameters. Replace them with the exact module and controller documentation for a real design.

The helper screens cold-corrected string Voc and array Isc. It does not perform all code-required protection or cable checks.

In [ ]:
string_check = check_pv_string_configuration(
    module_voc_v=49.5,
    modules_in_series=5,
    modules_in_parallel=3,
    minimum_ambient_temperature_c=5,
    voc_temperature_coefficient_per_c=-0.0028,
    controller_max_pv_voltage_v=300,
    module_isc_a=13.0,
    controller_max_input_current_a=50.0,
)
display(pd.Series(string_check).to_frame("screening_result"))

## 5. Simulate one day of PV-load-battery operation

Sizing from daily energy alone cannot show when the battery charges or discharges. This simplified hourly dispatch example lets PV serve the load first, charge the battery from surplus and discharge it when PV is insufficient.

The synthetic PV profile is a smooth daytime shape, not a weather-derived production profile. It ignores clouds, temperature, inverter maps, battery C-rate curves and degradation.

In [ ]:
hours = np.arange(24)
load_kw = load / 1000.0
pv_kw = 1.6 * np.maximum(0.0, np.sin((hours - 6) * np.pi / 12))

dispatch = simulate_hourly_dispatch(
    load_kw=load_kw,
    pv_generation_kw=pv_kw,
    battery_capacity_kwh=design["battery_nominal_kwh"],
    initial_soc_fraction=0.80,
    minimum_soc_fraction=1.0-cfg["depth_of_discharge"],
    charge_efficiency=0.95,
    discharge_efficiency=0.95,
    max_charge_kw=1.0,
    max_discharge_kw=1.0,
)
display(dispatch.round(3))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hours, load_kw, label="Load (kW)")
ax.plot(hours, pv_kw, label="Synthetic PV (kW)")
ax.plot(hours, dispatch["battery_discharge_to_load_kw"], label="Battery discharge (kW)")
ax.set(title="Illustrative hourly energy balance", xlabel="Hour of day", ylabel="Power (kW)")
ax.set_xticks(range(0,24,2))
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

print("Served energy (kWh):", round(dispatch.attrs["served_energy_kwh"], 3))
print("Unserved energy (kWh):", round(dispatch.attrs["unserved_energy_kwh"], 3))
print("Curtailed PV energy (kWh):", round(dispatch.attrs["curtailed_energy_kwh"], 3))

## 6. Preliminary lifecycle cost of energy

LCOE divides discounted lifetime cost by discounted lifetime electricity delivered. The following CAPEX, O&M, replacement cost and energy values are illustrative placeholders, not Bangladesh market quotations. Replace them with vendor quotes, project costs and a justified discount-rate basis.

In [ ]:
lcoe = levelized_cost_of_energy(
    initial_cost=450000,                  # illustrative currency units
    annual_om_cost=7000,
    annual_energy_kwh=4000,
    years=20,
    discount_rate=0.08,
    replacement_costs={10: 80000},
    annual_energy_degradation=0.005,
)
print("Illustrative LCOE (currency/kWh):", round(lcoe, 3))

## 7. Engineering review before using the design

The script is a screening aid. Before construction or procurement, verify worst-month energy supply, site shading, temperature-corrected module string limits, battery chemistry and C-rate, inverter surge behaviour, controller limits, cable ampacity, cable voltage drop, short-circuit protection, isolation, grounding, lightning/surge protection, structural load and applicable standards.

A full design should use measured load profiles and a chronological solar-resource series rather than this representative synthetic day.